<a href="https://colab.research.google.com/github/AhmedMahmoud-123/FlyRank_AI/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os

REPO_URL = 'https://github.com/AhmedMahmoud-123/FlyRank_AI.git'
REPO_DIR = 'FlyRank_AI'

if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL}
os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

Working directory: /content/FlyRank_AI


In [2]:
%pip -q install duckdb huggingface_hub

In [3]:
import os, getpass

HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

Paste your Hugging Face READ token (hf_...): ··········


In [4]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':       f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':       f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')

dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis:** one row = one content item (`content_hash_id`), aggregated across a
trailing performance window — NOT a page-day. The daily fact table (`fact_content_daily_performance`)
is report_date × client × content, but my lane rolls it up to one row per content item before
modeling (same grain as `w02_ml_task_framing`'s `is_declining_label`).

**Time window:** two adjacent 30-day windows, both ending at the panel's max `report_date`:
- `prev30` = days 31–60 back → feature inputs only
- `last30` = most recent 30 days → label input only (never a feature — that's the leakage
  notebook 02 warns about)

Client history is an *unbalanced panel* (`dim_clients.gsc_data_start` differs per client), so
this window is defined relative to each row's own data, not one global calendar date.

In [5]:
bounds = con.sql(f"SELECT MIN(report_date) AS min_d, MAX(report_date) AS max_d FROM {TABLES['fact_daily']}").df()
print(bounds)

# confirm grain: one row per (client, content, date) in the daily fact
dupes = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) c
    FROM {TABLES['fact_daily']}
    GROUP BY 1,2,3 HAVING c > 1 LIMIT 5
""").df()
print('duplicate (client, content, date) rows:', len(dupes))  # expect 0

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

       min_d      max_d
0 2025-01-27 2026-06-30


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

duplicate (client, content, date) rows: 5


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

| Bucket | Fields | Why |
|---|---|---|
| **Feature** | `imp_prev30`, `pos_last30`, `pos_volatility_last30`, `visible_queries`, `rare_share`, `anon_share`, `top_query_share` | Knowable before the label window closes |
| **Label** | `is_declining_label` (derived from `trend_direction == "down"`, i.e. `imp_last30 < 0.8 * imp_prev30`) | The thing I'm predicting |
| **Context** | `client_hash_id`, `content_hash_id` | Grouping / joins / GroupShuffleSplit — never model inputs |
| **Excluded** | `imp_last30`, `clk_last30` (label period — leakage), `competition_level` (fixed metadata, not a decision-improving target — per `w02_ml_task_framing`), `provider_used`/`model_used` (not signal, per data dictionary) | Each excluded for a specific reason, not just unused |

In [6]:
# feature_cols must never overlap the label window
feature_cols = ['imp_prev30', 'pos_last30', 'pos_volatility_last30',
                'visible_queries', 'rare_share', 'anon_share', 'top_query_share']
label_inputs = ['imp_last30']
assert not set(feature_cols) & set(label_inputs), "leakage: a feature is also a label input"
print("no overlap between features and label inputs — OK")

no overlap between features and label inputs — OK


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [7]:
missing = con.sql(f"""
    WITH per_content AS (
        SELECT content_hash_id,
               ANY_VALUE(content_visible_query_count) IS NULL AS missing_flag
        FROM {TABLES['fact_query_90d']}
        GROUP BY content_hash_id
    )
    SELECT missing_flag, COUNT(*) AS c
    FROM per_content
    GROUP BY missing_flag
""").df()
print(missing)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   missing_flag       c
0         False  133852


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Missing values / what this data can't tell you:**
- History depth differs per client — an unbalanced panel. A row's "prev30" window may sit
  before that client's `gsc_data_start`, silently producing zero-filled features that look
  like real zeros.
- Rows before a client's `ga4_data_start` have GA4 columns zero-filled with
  `ga4_data_available = FALSE` — must filter on the flag, not treat zeros as "no engagement."
- `fact_content_query_90d` is a fixed 90-day window that overlaps the last30 label period —
  its `*_last30`-style columns are leakage; only `*_prev30`-safe aggregates go in as features.
- Keyword-context missingness (search_volume, competition, cpc) follows `content_type`, not
  randomness — e.g. `feedly article` rows carry no keyword data at all.

**Output:** a ranked list of content items scored by predicted decline risk
(`is_declining_label` probability), handed to a human reviewer as directional
decision-support — never framed as "predicting Google's algorithm."

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.